In [3]:
import pandas as pd
import numpy as np
import joblib

try:
    model_pack = joblib.load('../models/profit_prediction_engine_HGBRegressor_final.pkl')
    print("✅ Modelul a fost incarcat cu succes.")
except FileNotFoundError:
    print("❌ Eroare: Fisierul .pkl nu a fost gasit. Verifica numele!")
    raise

new_data = pd.DataFrame([{
    'sales': 1500.0,
    'discount': 0.15,
    'quantity': 50,
    'log_sales': np.log1p(1500.0),
    'unit_sales': 1500.0 / (50 + 1e-5),
    'discount_sq': 0.15 ** 2,
    'high_disc_pct': 0.0,
    'zero_disc_pct': 0.0,
    'high_discount': 0,
    'zero_discount': 0,
    'risk_subcat_region': 50.0,
    'risk_category': 80.0,
    'risk_product': 40.0,
    'seasonal_risk': 60.0,
    'cust_total_profit': 5000.0,
    'cust_avg_profit': 120.0,
    'cust_avg_margin': 0.18,
    'cust_n_orders': 20,
    'cust_loss_rate': 0.10,
    'cust_recency': 1,
    'cust_tenure': 3,
    'cust_has_history': 1,
    'cust_high_disc_rate': 0.05,
    'is_high_season': 0,
    'is_q1': 0,
    'shipping_delay': 2.5,
    'ship_profit_avg': 55.0,
    'ship_mode_enc': 0,
    'segment_enc': 0,
    'p_last': 0.15,
    'profit_last': 225.0,
    'sales_last': 1200.0,
    'margin_trend': 0.0,
    'discount_efficiency': 0.15 / (0.15 + 0.01),
    'high_loss_risk': 0,
    'n_orders': 5,
    'n_products': 3,
    'product_diversity': 3 / (5 + 1e-5),
    'Art_West': 0,
    'Acc_East': 0,
}])

p_margin_est = 0.10
new_data['margin_trend'] = p_margin_est - new_data['p_last']

new_data_clust = new_data[['sales', 'discount', 'cust_avg_margin', 'cust_loss_rate']].copy()
new_data_clust.insert(1, 'p_margin', p_margin_est)

X_new_clust = model_pack['robust_scaler'].transform(
    new_data_clust[['sales', 'p_margin', 'discount', 'cust_avg_margin', 'cust_loss_rate']]
)
segment = model_pack['kmeans_model'].predict(X_new_clust)[0]
print(f"\n--- TESTARE PRODUCTIE ---")
print(f"Identificare Segment: {segment}")

features = model_pack['features_list']
meta_features = model_pack['meta_features']
alpha = model_pack['blend_alpha']
pt = model_pack['power_transformer']

new_data['segment_client'] = segment

missing = [f for f in meta_features if f not in new_data.columns]
if missing:
    print(f"⚠ Features lipsa: {missing}")

if segment in model_pack['experts']:
    expert_model = model_pack['experts'][segment]
    pred_trans_ex = expert_model.predict(new_data[features])
    pred_margin_ex = pt.inverse_transform(
        pd.DataFrame(pred_trans_ex, columns=['margin_w'])
    ).ravel()
    expert_profit = pred_margin_ex[0] * new_data['sales'].values[0]
else:
    print(f"⚠ Nu exista expert pt segmentul {segment} — folosim doar meta-modelul")
    pred_margin_ex = np.array([0.0])
    expert_profit = 0.0
    alpha = 0.0

meta_model = model_pack['meta_model']
pred_trans_mt = meta_model.predict(new_data[meta_features])
pred_margin_mt = pt.inverse_transform(
    pd.DataFrame(pred_trans_mt, columns=['margin_w'])
).ravel()
meta_profit = pred_margin_mt[0] * new_data['sales'].values[0]

final_profit = alpha * expert_profit + (1 - alpha) * meta_profit
final_margin = alpha * pred_margin_ex[0] + (1 - alpha) * pred_margin_mt[0]

print(f"profit Experts:    ${expert_profit:.2f}  (marja {pred_margin_ex[0] * 100:.2f}%)")
print(f"profit Meta:      ${meta_profit:.2f}  (marja {pred_margin_mt[0] * 100:.2f}%)")
print(f"Combinatia Alpha:       {alpha:.2f} × expert + {1 - alpha:.2f} × meta")
print(f"Marja prezisa:     {final_margin * 100:.2f}%")
print(f"Profit final prezis:     ${final_profit:.2f}")
print("-" * 25)

✅ Modelul a fost incarcat cu succes.

--- TESTARE PRODUCTIE ---
Identificare Segment: 0
profit Experts:    $71.37  (marja 4.76%)
profit Meta:      $64.92  (marja 4.33%)
Combinatia Alpha:       0.50 × expert + 0.50 × meta
Marja prezisa:     4.54%
Profit final prezis:     $68.14
-------------------------
